# 🧬 Pipeline de Análisis GC-MS Completo

## ⚠️ **CÓMO USAR ESTE NOTEBOOK**

### ✅ FLUJO CORRECTO (recomendado):

1. **Ejecuta celda 1** (Setup) - una sola vez
2. **Ejecuta celda 2** (Selector) - verás los botones para seleccionar archivos
3. **Haz clic en "+ Buscar archivos .cdf"** en celda 2
4. **Selecciona los archivos** que quieras analizar desde el explorador
5. **Haz clic en "▶ Confirmar y analizar"**
6. **¡Listo!** El pipeline se ejecuta automáticamente y genera:
   - ✅ Análisis de TODAS las muestras seleccionadas
   - ✅ Matriz consolidada de features
   - ✅ PCAs (si 2+ muestras)
   - ✅ Heatmaps (si 2+ muestras)
   - ✅ Reporte XLSX con múltiples hojas
   - 📁 Guardado en: `output/report/` y `output/figures/` (POR AHORA NO FUNCIONA)

### ⚠️ **NO EJECUTES** manualmente: (POR AHORA EJECUCIÓN DE ESTAS MANUALMENTE)
- ❌ Celdas 5-19 (son solo para demostración de detalle de UNA muestra)
- ❌ Celdas 20+ (se ejecutan automáticamente)

---

## 📊 Qué hace cada parte:

| Sección | Propósito | Ejecutar | Resultado |
|---------|-----------|----------|-----------|
| **Celda 1** | Setup + imports | 🟢 Manual (1x) | Prepara el notebook |
| **Celda 2** | Selector de archivos | 🟢 Manual | Interfaz para seleccionar archivos |
| **Celdas 5-19** | Demostración de 1 muestra | ⚫ Automático | Gráficos de detalle (demo) |
| **Celdas 17+20** | NIST + Pipeline completo | ✅ Automático al confirmar | PCAs, heatmaps, reporte XLSX |

---

## 🚨 Si algo falla:

1. Verifica que los archivos `.cdf` sean válidos
2. Comprueba que haya espacio en disco
3. Mira el mensaje de error en la salida
4. Reinicia el kernel y repite (menú: Kernel → Restart)



In [ ]:
# Setup automático: asegurar que el proyecto esté en sys.path y verificar imports
import sys, os
print('Current cwd:', os.getcwd())
# Añadir candidatos de raíz del proyecto
candidates = [
    os.path.abspath(os.path.join(os.getcwd(), '..')),
    os.path.abspath(os.path.join(os.getcwd(), '..', '..')),
    os.path.abspath(os.getcwd())
]
for p in candidates:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
        print('Añadido a sys.path:', p)

# Verificar imports críticos
missing = []
try:
    import pipeline
    print('Import pipeline: OK ->', getattr(pipeline, '__file__', '?'))
except Exception as e:
    print('Import pipeline: ERROR ->', type(e).__name__, e)
    missing.append('pipeline')
try:
    import ipywidgets
except Exception as e:
    print('Import ipywidgets: ERROR ->', type(e).__name__, e)
    missing.append('ipywidgets')

if missing:
    print('Faltan módulos o imports:', missing)
    print('Ejecuta la celda de instalación o instala los paquetes necesarios.')


In [ ]:
# SELECTOR DE ARCHIVOS .CDF
# Selecciona los archivos → pulsa "Confirmar" → el pipeline arranca en la celda siguiente
from IPython.display import display, HTML, clear_output
from ipywidgets import SelectMultiple, Button, VBox, HBox, Label, Output
import IPython, pipeline, config
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

original_list_cdf  = pipeline.list_cdf_files
selected_cdf_paths = []
out_run = Output()

# ── Widgets ──────────────────────────────────────────────────────────────────
existing_list = SelectMultiple(rows=8, description='', layout={'width': '340px'})
selected_list = SelectMultiple(rows=8, description='', layout={'width': '340px'})
browse_btn  = Button(description='+ Buscar archivos .cdf', button_style='info',    layout={'width': '200px'})
add_btn     = Button(description='Añadir →',               button_style='primary', layout={'width': '110px'})
remove_btn  = Button(description='← Quitar',               button_style='warning', layout={'width': '110px'})
confirm_btn = Button(description='▶ Confirmar y analizar', button_style='success', layout={'width': '200px'})
reset_btn   = Button(description='↺ Reiniciar',            button_style='',        layout={'width': '120px'})
status_lbl  = Label(value='')

def _refresh_existing():
    try:    paths = list(original_list_cdf())
    except: paths = []
    existing_list.options = [(p.name, str(p)) for p in paths]
_refresh_existing()

def _on_browse(b):
    """Abre el explorador y añade los archivos elegidos DIRECTAMENTE a Selected."""
    try:
        root = tk.Tk(); root.withdraw(); root.wm_attributes('-topmost', True)
        paths = filedialog.askopenfilenames(
            title='Selecciona archivos .cdf para analizar',
            filetypes=[('CDF files', '*.cdf *.CDF'), ('Todos', '*.*')])
        root.destroy()
    except Exception as e:
        status_lbl.value = f'Error explorador: {e}'; return

    if not paths:
        status_lbl.value = 'No se seleccionó ningún archivo.'; return

    curr_sel = dict(selected_list.options)
    added = 0
    for p in paths:
        src = Path(p)
        if str(src) not in curr_sel.values():
            curr_sel[src.name] = str(src)
            added += 1

    selected_list.options = list(curr_sel.items())
    status_lbl.value = f'{added} archivo(s) añadido(s) → {len(curr_sel)} en Selected.'

def _on_add(b):
    """Mueve los items seleccionados en Existing → Selected."""
    vals = list(existing_list.value)
    if not vals: status_lbl.value = 'Selecciona primero en la lista izquierda.'; return
    curr = dict(selected_list.options)
    for v in vals:
        if v not in curr.values(): curr[Path(v).name] = v
    selected_list.options = list(curr.items())
    status_lbl.value = f'{len(selected_list.options)} archivo(s) en Selected.'

def _on_remove(b):
    to_remove = set(selected_list.value)
    selected_list.options = [(n, v) for n, v in selected_list.options if v not in to_remove]
    status_lbl.value = f'{len(selected_list.options)} archivo(s) en Selected.'

def _on_reset(b):
    global selected_cdf_paths
    selected_cdf_paths = []
    pipeline.list_cdf_files = original_list_cdf
    IPython.get_ipython().user_ns.pop('selected_cdf_paths', None)
    selected_list.options = []
    for w in [browse_btn, add_btn, confirm_btn, existing_list, selected_list]: w.disabled = False
    with out_run: clear_output()
    _refresh_existing()
    status_lbl.value = 'Reiniciado.'

def _on_confirm(b):
    global selected_cdf_paths
    opts = list(selected_list.options)
    if not opts: status_lbl.value = '⚠ Añade archivos a Selected primero.'; return

    selected_cdf_paths = [Path(v) for _, v in opts]
    pipeline.list_cdf_files = lambda data_dir=None: selected_cdf_paths
    # Exponer en el namespace del kernel
    IPython.get_ipython().user_ns['selected_cdf_paths'] = selected_cdf_paths

    for w in [browse_btn, add_btn, confirm_btn, existing_list, selected_list]: w.disabled = True
    status_lbl.value = f'⏳ {len(selected_cdf_paths)} archivo(s) — ejecutando análisis…'

    with out_run:
        clear_output(wait=True)
        print(f"⏳ Ejecutando análisis de {len(selected_cdf_paths)} archivo(s)...\n")
        ok = _exec_cell('cell-preprocess')   # Celda: Análisis Completo de Muestras
        if ok:
            status_lbl.value = f'✔ Análisis completado — {len(selected_cdf_paths)} muestra(s).'
        else:
            status_lbl.value = '❌ Error en el análisis (ver output arriba).'

def _exec_cell(cell_id):
    import json, os
    ip = IPython.get_ipython()
    ns = ip.user_ns

    nb_path = ns.get('__vsc_ipynb_file__', None)
    if not nb_path:
        hits = sorted(Path(os.getcwd()).glob('*.ipynb'))
        nb_path = str(hits[0]) if hits else None

    if not nb_path or not Path(nb_path).exists():
        print(f'⚠ No se localizó el notebook.')
        print(f'  Ejecuta manualmente la celda "Análisis Completo de Muestras".')
        status_lbl.value = '⚠ Ejecuta manualmente la celda de análisis.'
        return False

    with open(nb_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    for cell in nb['cells']:
        cell_id_clean    = cell.get('id', '').lstrip('#')
        search_id_clean  = cell_id.lstrip('#')
        if cell_id_clean == search_id_clean and cell['cell_type'] == 'code':
            src = ''.join(cell['source']).strip()
            if not src:
                return True
            result = ip.run_cell(src)
            return not (result.error_before_exec or result.error_in_exec)

    print(f'⚠ Celda "{cell_id}" no encontrada en el notebook.')
    return False

browse_btn.on_click(_on_browse); add_btn.on_click(_on_add); remove_btn.on_click(_on_remove)
confirm_btn.on_click(_on_confirm); reset_btn.on_click(_on_reset)

display(HTML('<h3>📂 Seleccionar archivos .cdf para el análisis</h3>'))
display(HTML('<i>Usa "+ Buscar archivos .cdf" para seleccionar archivos desde cualquier carpeta. '
             'Los archivos de la carpeta por defecto aparecen en el panel izquierdo.</i>'))
display(browse_btn)
display(HBox([
    VBox([Label('📂 Carpeta por defecto (opcional)'), existing_list]),
    VBox([add_btn, remove_btn], layout={'justify_content': 'center', 'margin': '40px 12px'}),
    VBox([Label('✅ A analizar (Selected)'), selected_list]),
]))
display(HBox([confirm_btn, reset_btn]))
display(status_lbl)
display(HTML('<hr style="margin-top:12px"><b>Output del análisis:</b>'))
display(out_run)


# AutoGCMS Pipeline

Pipeline automatizado para procesamiento de datos GC-MS.

**Flujo:**
1. **Exploración** (una muestra): preprocesado, detección de picos, espectros, identificación NIST
2. **Pipeline completo** (todas las muestras): módulos 1-6 automáticos → `final_report/`

**Requisito:** Exportar MAINLIB desde NIST MS Search a formato .msp y configurar la ruta en `config.py` (`NIST_MSP_PATH`).

---

## 1. Seleccionar archivo .cdf

Cambia `FILE_IDX` para procesar otra muestra.

In [ ]:
import sys
from pathlib import Path

# Añadir el directorio padre al path de Python para encontrar los módulos
notebook_dir = Path.cwd()
parent_dir = notebook_dir.parent
sys.path.insert(0, str(parent_dir))

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import config
import traceback
from pipeline import list_cdf_files, run_single_sample
from preprocessing import preprocess
from peak_detection import find_and_filter_peaks
from spectra import extract_all_spectra, export_msp
from visualization import plot_tic_preprocessing, plot_peaks, plot_peak_quality, plot_spectrum
from nist_search import match_all_spectra, load_nist_library
from matrix_builder import align_peaks, build_data_matrix, build_feature_metadata, export_matrix, export_xlsx_report
import qc

print("✓ Importaciones cargadas correctamente")

---
## 2. Análisis Completo de Todas las Muestras (Módulos 1-6)

Procesa TODOS los archivos seleccionados: preprocesado → picos → espectros → NIST → matriz consolidada

In [ ]:
import sys, traceback
import pandas as pd
import numpy as np
import IPython
from pathlib import Path

# ── sys.path ──────────────────────────────────────────────────────────────────
for _p in [Path.cwd().parent, Path.cwd()]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# ── Imports del proyecto ──────────────────────────────────────────────────────
import config
from preprocessing import preprocess
from peak_detection import find_and_filter_peaks
from spectra import extract_all_spectra
from nist_search import match_all_spectra, load_nist_library

print(f"\n{'='*100}")
print(f"ANÁLISIS COMPLETO DE MUESTRAS SELECCIONADAS")
print(f"{'='*100}\n")

# ── Obtener archivos seleccionados ────────────────────────────────────────────
_ns       = IPython.get_ipython().user_ns
cdf_files = _ns.get('selected_cdf_paths', None)

if not cdf_files:
    print("⚠️  Sin archivos seleccionados en esta sesión.")
    print("    → Ejecuta la celda del SELECTOR (celda 2),")
    print("      añade archivos al panel 'A analizar (Selected)'")
    print("      y pulsa '▶ Confirmar y analizar'.")
else:
    print(f"Analizando {len(cdf_files)} archivo(s):")
    for i, f in enumerate(cdf_files, 1):
        print(f"  {i}. {Path(f).name}")

    # ── Cargar librería NIST ──────────────────────────────────────────────────
    library = _ns.get('library') or _ns.get('_nist_library_cache')
    if not library:
        print(f"\n⚠️ Librería NIST no cargada. Cargando ahora...")
        try:
            from nist_binary_reader import try_load_nist_mainlib_binary
            nist_dir = Path(r"C:\Users\LENOVO\OneDrive\Documentos\MATLAB\MSSEARCH")
            library  = try_load_nist_mainlib_binary(nist_dir, max_attempts=5000)
            if not library:
                all_msp = list(nist_dir.rglob("*.msp")) + list(nist_dir.rglob("*.MSP"))
                big_msp = sorted([f for f in all_msp if f.stat().st_size > 100_000],
                                  key=lambda f: f.stat().st_size, reverse=True)
                library = load_nist_library(str(big_msp[0])) if big_msp else load_nist_library()
            if not library:
                library = []
            for spec in library:
                if "mz" not in spec or "intensities" not in spec:
                    spec["mz"]          = np.array([], dtype=np.float32)
                    spec["intensities"] = np.array([], dtype=np.float32)
            _ns['library'] = library
            print(f"✓ Librería NIST cargada: {len(library)} compuestos\n")
        except Exception as _e:
            print(f"✗ Error cargando NIST: {_e}\n")
            library = []

    # ── Procesar cada muestra ─────────────────────────────────────────────────
    all_sample_data     = {}
    peak_summary        = {}
    all_identifications = []

    print(f"\n{'='*100}")
    print(f"PROCESANDO MUESTRAS")
    print(f"{'='*100}\n")

    for idx, cdf_file in enumerate(cdf_files, 1):
        sample_name = Path(cdf_file).stem
        print(f"[{idx}/{len(cdf_files)}] ANALIZANDO: {sample_name}")
        print("-" * 100)
        try:
            data  = preprocess(cdf_file)
            print(f"  ✓ Preprocesado: {data.n_scans} scans, "
                  f"{data.scan_times_min[0]:.2f}-{data.scan_times_min[-1]:.2f} min, "
                  f"m/z {data.mass_range[0]:.1f}-{data.mass_range[1]:.1f}")

            peaks = find_and_filter_peaks(data)
            print(f"  ✓ Picos detectados: {peaks.n_peaks}")

            spectra    = extract_all_spectra(data, peaks)
            n_coeluted = sum(1 for s in spectra if s.is_coeluted)
            print(f"  ✓ Espectros extraídos: {len(spectra)} (coeluidados: {n_coeluted})")

            if library:
                matches   = match_all_spectra(spectra, library, top_n=5,
                                              threshold=0.3, sample_name=sample_name)
                rank1_ids = sum(1 for m in matches if m.candidates)
                print(f"  ✓ Identificación NIST: {rank1_ids} picos identificados (umbral 0.3)")
            else:
                print(f"  ⚠ Sin librería NIST (identificación saltada)")
                matches = []

            all_sample_data[sample_name] = {
                'data': data, 'peaks': peaks, 'spectra': spectra, 'matches': matches
            }
            peak_summary[sample_name] = peaks.n_peaks

            for spectrum_idx, match_result in enumerate(matches):
                spectrum = spectra[spectrum_idx]
                if not match_result.candidates:
                    all_identifications.append({
                        'sample': sample_name, 'peak_id': spectrum.peak_id,
                        'rt_min': spectrum.rt_min, 'compound': 'No match', 'score': 0.0,
                    })
                else:
                    top_match = match_result.candidates[0]
                    lib_idx   = top_match.get('library_idx', 0)
                    compound  = (library[lib_idx].get('name', 'Unknown')
                                 if lib_idx < len(library) else 'Unknown')
                    all_identifications.append({
                        'sample':   sample_name,
                        'peak_id':  spectrum.peak_id,
                        'rt_min':   spectrum.rt_min,
                        'compound': compound,
                        'score':    top_match.get('score', 0.0),
                    })

            # Exponer última muestra en namespace para celdas de visualización
            _ns.update({'data': data, 'peaks': peaks,
                        'spectra': spectra, 'matches': matches})
            print(f"  ✓ Completada exitosamente\n")

        except Exception as _e:
            print(f"  ✗ Error: {_e}")
            traceback.print_exc()
            print()

    # ── Resumen ───────────────────────────────────────────────────────────────
    print(f"{'='*100}")
    print(f"RESUMEN")
    print(f"{'='*100}")
    print(f"\nMuestras procesadas: {len(all_sample_data)}/{len(cdf_files)}")
    for name, count in peak_summary.items():
        print(f"  • {name}: {count} picos")

    if len(all_sample_data) == 0:
        print(f"\n✗ Ninguna muestra se procesó correctamente. Revisa los archivos .cdf.")
    else:
        _ns.update({
            'all_sample_data':     all_sample_data,
            'peak_summary':        peak_summary,
            'all_identifications': all_identifications,
        })
        print(f"\n✓ Datos listos. Variables: all_sample_data, peak_summary, all_identifications.")


In [ ]:
from visualization import plot_tic_preprocessing
fig = plot_tic_preprocessing(data)
plt.show()

---
## 3. Deteccion de picos (Modulo 2)

In [ ]:
from peak_detection import find_and_filter_peaks

peaks = find_and_filter_peaks(data)
print(f"Picos detectados: {peaks.n_peaks}")
print(f"  (peak_id = ranking por intensidad, scan_idx = indice real en TIC)")
print(f"\nTop 20 picos por intensidad:")
peaks.table.head(20)[['peak_id', 'scan_idx', 'rt_min', 'intensity', 'snr', 'width_sec']]

In [ ]:
from visualization import plot_peaks, plot_peak_quality

fig = plot_peaks(data, peaks)
plt.show()

fig = plot_peak_quality(peaks)
plt.show()

---
## 4. Extraccion de espectros de masas (Modulo 3)

In [ ]:
from spectra import extract_all_spectra, export_msp

spectra = extract_all_spectra(data, peaks)
n_coeluted = sum(1 for s in spectra if s.is_coeluted)
n_subpeaks = sum(1 for s in spectra if s.subpeak > 0)
print(f"Espectros extraidos: {len(spectra)}")
print(f"  Picos simples: {len(spectra) - n_subpeaks}")
print(f"  Picos co-eluidos (Gaussiana BIC): {n_coeluted}")
print(f"  Subpicos generados: {n_subpeaks}")

# Exportar a .msp
msp_path = config.OUTPUT_DIR / f"{data.name}_spectra.msp"
export_msp(spectra, msp_path, sample_name=data.name)
print(f"\nExportados a: {msp_path}")

In [ ]:
from visualization import plot_spectrum

# Espectro del pico mas intenso (subpeak=0 = pico simple)
simple_spectra = [s for s in spectra if s.subpeak == 0]
sp = simple_spectra[0] if simple_spectra else spectra[0]

sub_label = f" (subpeak {sp.subpeak})" if sp.subpeak > 0 else ""
fig = plot_spectrum(
    sp.mz, sp.intensities,
    title=f"Espectro de masas -- Pico {sp.peak_id}{sub_label} (RT={sp.rt_min:.2f} min)"
)
plt.show()

print(f"\nFragmentos principales (m/z : intensidad):")
for idx in sp.intensities.argsort()[-10:][::-1]:
    print(f"  m/z {sp.mz[idx]:6.1f} : {sp.intensities[idx]:10.0f}")

---
## 5. Identificacion de compuestos (Modulo 4) -- Libreria NIST

Compara cada espectro contra la libreria NIST (cargada en el setup) usando similitud coseno.  
Devuelve los top N candidatos por espectro con score >= umbral.

In [ ]:
print(f"\n{'='*100}")
print(f"CARGANDO LIBRERÍA NIST MAINLIB")
print(f"{'='*100}\n")

nist_dir = Path(r"C:\Users\LENOVO\OneDrive\Documentos\MATLAB\MSSEARCH")

# ──── INTENTAR 1: LEER MAINLIB BINARIO DIRECTAMENTE ────
print("⚙️  Intento 1: Leer MAINLIB binario directamente...")
library = None
try:
    from nist_binary_reader import try_load_nist_mainlib_binary
    library = try_load_nist_mainlib_binary(nist_dir, max_attempts=5000)
except Exception as e:
    print(f"❌ Error: {e}")
    library = None

if not library:
    print("❌ No se pudo leer MAINLIB binario\n")
    
    # ──── INTENTAR 2: .msp con contenido────
    print("⚙️  Intento 2: Buscar archivos .msp locales...")
    
    all_msp_files = list(nist_dir.rglob("*.msp")) + list(nist_dir.rglob("*.MSP"))
    msp_with_content = [f for f in all_msp_files if f.stat().st_size > 100_000]
    
    if msp_with_content:
        msp_by_size = sorted(msp_with_content, key=lambda f: f.stat().st_size, reverse=True)
        best_msp = msp_by_size[0]
        print(f"✅ Encontrado: {best_msp.name} ({best_msp.stat().st_size / (1024*1024):.1f} MB)")
        try:
            library = load_nist_library(str(best_msp))
        except Exception as e:
            print(f"❌ Error cargando .msp: {e}")
            library = None
    else:
        print("❌ No hay .msp con contenido\n")

# ──── INTENTO 3: Fallback a 17 compuestos default ────
if not library:
    print("⚙️  Intento 3: Usar librería default (17 compuestos)...")
    try:
        library = load_nist_library()
    except Exception as e:
        print(f"❌ Error cargando librería default: {e}")
        library = []

# ─── VALIDAR Y NORMALIZAR ───
if not library:
    library = []

# Normalizar espectros que vengan del lector binario (add mz/intensities si faltan)
for spec in library:
    if "mz" not in spec or "intensities" not in spec:
        spec["mz"] = np.array([], dtype=np.float32)
        spec["intensities"] = np.array([], dtype=np.float32)

# ──── RESULTADO FINAL ────
print(f"\n{'='*100}")
print(f"✅ LIBRERÍA CARGADA EXITOSAMENTE")
print(f"{'='*100}")
print(f"📊 Compuestos de referencia: {len(library)}")
if len(library) > 0:
    print(f"🔬 Espectros a buscar: {len(spectra) if 'spectra' in locals() else 'N/A'}")
    print(f"⚙️  Umbral coseno: {config.COSINE_THRESHOLD}")
    print(f"🏆 Top candidatos: {config.NIST_TOP_N}")
    # Mostrar información de primeros 3
    print(f"\nPrimeros 3 compuestos en librería:")
    for i, comp in enumerate(library[:3]):
        mz_count = len(comp.get("mz", []))
        formula = comp.get("formula", "")
        print(f"  {i+1}. {comp.get('name', 'Unknown'):40s} | m/z picos: {mz_count:3d} | {formula}")
else:
    print("\n⚠️  librería vacía! Los identificaciones NIST no funcionarán.")

In [ ]:
# DEBUG: Ver TODOS los compuestos en la librería
print(f"\n{'='*100}")
print(f"DEBUG: TODOS LOS COMPUESTOS EN LA LIBRERÍA NIST")
print(f"{'='*100}\n")

for i, comp in enumerate(library):
    name = comp.get('name', 'NO NAME')
    formula = comp.get('formula', '')
    cas = comp.get('cas', '')
    
    # Detectar si es un compuesto real o de demostración
    is_demo = 'Scan' in name or 'DEMO' in name
    marker = "⚠️ DEMO" if is_demo else "✅ REAL"
    
    print(f"{i:2d}. [{marker}] {name}")
    if formula:
        print(f"    Formula: {formula}, CAS: {cas}")


In [ ]:
# Matching contra libreria NIST usando similitud coseno
from nist_search import match_all_spectra
import pandas as pd

print(f"\n{'='*100}")
print(f"IDENTIFICACIÓN DE COMPUESTOS CONTRA LIBRERÍA NIST")
print(f"{'='*100}")
print(f"Librería: {len(library)} compuestos (15 reales + 2 de demo)")
print(f"Espectros a buscar: {len(spectra)}")
print(f"Umbral de similitud: 0.3\n")

threshold_for_matching = 0.3

matches = match_all_spectra(
    spectra, library,
    top_n=5,
    threshold=threshold_for_matching,
    sample_name=data.name,
)

# Construir tabla con TODOS los matches
results = []
for spectrum_idx, match_result in enumerate(matches):
    spectrum = spectra[spectrum_idx]
    
    if not match_result.candidates:
        # Sin matches
        results.append({
            'peak_id': spectrum.peak_id,
            'subpeak': spectrum.subpeak,
            'rt_min': spectrum.rt_min,
            'rank': 1,
            'compound_name': 'No match',
            'formula': '',
            'cas': '',
            'match_score': 0.0,
            'n_matched_peaks': 0,
            'library_type': 'NONE'
        })
    else:
        # Con matches
        for rank, candidate in enumerate(match_result.candidates, 1):
            lib_idx = candidate.get('library_idx', None)
            score = candidate.get('score', 0.0)
            n_peaks = candidate.get('n_matched_peaks', 0)
            
            # Obtener datos del compuesto de la librería
            if lib_idx is not None and 0 <= lib_idx < len(library):
                lib_compound = library[lib_idx]
                compound_name = lib_compound.get('name', 'Unknown')
                formula = lib_compound.get('formula', '')
                cas = lib_compound.get('cas', '')
                
                # Identificar si es compuesto real o demo
                if 'Scan' in compound_name or 'DEMO' in compound_name:
                    library_type = 'DEMO'
                else:
                    library_type = 'REAL'
            else:
                compound_name = 'Unknown'
                formula = ''
                cas = ''
                library_type = 'UNKNOWN'
            
            results.append({
                'peak_id': spectrum.peak_id,
                'subpeak': spectrum.subpeak,
                'rt_min': spectrum.rt_min,
                'rank': rank,
                'compound_name': compound_name,
                'formula': formula,
                'cas': cas,
                'match_score': score,
                'n_matched_peaks': n_peaks,
                'library_type': library_type
            })

id_table = pd.DataFrame(results)

# —— ESTADÍSTICAS ——
rank1_all = id_table[id_table['rank'] == 1]
n_real = len(rank1_all[rank1_all['library_type'] == 'REAL'])
n_demo = len(rank1_all[rank1_all['library_type'] == 'DEMO'])
n_no_match = len(rank1_all[rank1_all['compound_name'] == 'No match'])

print(f"{'='*100}")
print(f"RESULTADOS - TODAS LAS IDENTIFICACIONES")
print(f"{'='*100}")
print(f"✅ Compuestos REALES identificados: {n_real}")
print(f"⚠️  Matches a espectros DEMO: {n_demo}")
print(f"❌ Sin match: {n_no_match}")
print(f"📊 Total: {len(rank1_all)} picos\n")

# Guardar tabla
id_csv = config.OUTPUT_DIR / f"{data.name}_identifications.csv"
id_table.to_csv(id_csv, sep=';', encoding='utf-8-sig', index=False)
print(f"📁 Tabla guardada en: {id_csv}\n")

# Mostrar TODOS los picos identificados
cols_display = ['peak_id', 'subpeak', 'rt_min', 'rank', 'compound_name', 'formula',
                'match_score', 'n_matched_peaks', 'library_type']
                
# Ordenar por peak_id y luego por rank
display_table = id_table[id_table['rank'] == 1][cols_display].sort_values('peak_id')

print(f"{'='*100}")
print(f"🔬 TODOS LOS PICOS DETECTADOS Y SUS IDENTIFICACIONES (rank 1)")
print(f"{'='*100}\n")
display(display_table.reset_index(drop=True))

---
## 6. Pipeline completo: TODAS las muestras + Reporte final

Procesa automaticamente **todos** los archivos .cdf del directorio de datos:
- Modulos 1-4 por cada muestra (con libreria NIST)
- Modulo 5: alineamiento de picos entre muestras
- Modulo 6: genera `final_report/` con QC, PCA, heatmap y XLSX multi-hoja

In [ ]:
import traceback
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import IPython, config
from pathlib import Path

from pipeline import run_single_sample
from matrix_builder import (
    SampleResult, align_peaks,
    build_data_matrix, build_feature_metadata,
    export_matrix, export_xlsx_report,
)
import qc


def _run_pipeline():
    ns = IPython.get_ipython().user_ns

    # ─── Comprobar que hay archivos seleccionados ─────────────────────────────
    selected_cdf_paths = ns.get('selected_cdf_paths', [])
    if not selected_cdf_paths:
        print("ℹ Sin archivos .cdf seleccionados. Ejecuta primero la celda del selector.")
        return

    print(f"▶ Procesando {len(selected_cdf_paths)} archivo(s)...")

    # ─── Directorios de salida ────────────────────────────────────────────────
    output_dir  = Path(config.OUTPUT_DIR)
    report_dir  = output_dir / "report"
    figures_dir = output_dir / "figures"
    output_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    # ─── Cargar librería NIST (una vez) ──────────────────────────────────────
    nist_library = ns.get('_nist_library_cache', None)
    if nist_library is None:
        print("  Cargando librería NIST…")
        try:
            from nist_search import load_nist_library
            nist_library = load_nist_library(config.NIST_MSP_PATH)
            ns['_nist_library_cache'] = nist_library
            print(f"  Librería NIST cargada: {len(nist_library)} espectros")
        except Exception as exc:
            print(f"  ⚠ No se pudo cargar la librería NIST: {exc}")
            nist_library = []

    # ═════════════════════════════════════════════════════════════════════════
    # PASO 1 — Procesar cada muestra
    # ═════════════════════════════════════════════════════════════════════════
    print("\n─── PASO 1: Procesamiento por muestra ───")
    sample_results = {}
    gcms_data_dict = {}

    for cdf_path in selected_cdf_paths:
        name = Path(cdf_path).stem
        print(f"  [{name}]")
        try:
            gcms_data, peaks, spectra, matches, _id_tbl = run_single_sample(
                cdf_path, nist_library
            )
            result = SampleResult(
                name=name,
                peak_table=peaks.table,
                spectra=spectra,
                matches=matches,
            )
            sample_results[name] = result
            gcms_data_dict[name] = gcms_data
            print(f"    {peaks.n_peaks} picos detectados")
        except Exception:
            print(f"    ✗ Error procesando {name}:")
            traceback.print_exc()

    if not sample_results:
        print("✗ Ninguna muestra procesó correctamente. Abortando.")
        return

    # ─── Inferir metadatos de muestras ────────────────────────────────────────
    sample_names = list(sample_results.keys())
    sample_meta  = qc.infer_sample_metadata(sample_names)
    print(f"\n  Tipos de muestra detectados:")
    print(sample_meta[["sample_id", "sample_type"]].to_string(index=False))

    # ═════════════════════════════════════════════════════════════════════════
    # PASO 1.5 — PCA de TIC (control de calidad técnico)
    # ═════════════════════════════════════════════════════════════════════════
    pca_tic_result = None
    if getattr(config, 'ENABLE_PCA_TIC', True) and len(gcms_data_dict) >= 2:
        print("\n─── PASO 1.5: PCA de TIC (QC técnico) ───")
        try:
            tic_matrix, _ = qc.build_tic_matrix(gcms_data_dict)
            ok, msg = qc.validate_minimum_data(
                len(tic_matrix), tic_matrix.shape[1],
                min_samples=getattr(config, 'MIN_SAMPLES_FOR_PCA', 3),
                min_features=2,
                name="PCA TIC",
            )
            if ok:
                pca_tic_result = qc.pca_tic_qc(tic_matrix)
                fig_tic = qc.plot_pca_tic(
                    pca_tic_result,
                    sample_metadata=sample_meta,
                    save_path=figures_dir / "pca_tic_qc.png",
                )
                if fig_tic:
                    plt.show()
                    plt.close(fig_tic)
                    print(f"  ✅ PCA TIC mostrado y guardado → {figures_dir / 'pca_tic_qc.png'}")
            else:
                print(f"  ⚠ {msg}")
        except Exception:
            print("  ⚠ Error en PCA TIC:")
            traceback.print_exc()

    # ═════════════════════════════════════════════════════════════════════════
    # PASO 2 — Alineamiento de picos y construcción de la matriz
    # ═════════════════════════════════════════════════════════════════════════
    print("\n─── PASO 2: Alineamiento y construcción de la matriz ───")
    features     = align_peaks(sample_results)
    data_matrix  = build_data_matrix(features, sample_names)
    feat_meta_df = build_feature_metadata(features)

    print(f"  Matriz: {data_matrix.shape[0]} muestras × {data_matrix.shape[1]} features")
    print(f"  Features con ID NIST: {(feat_meta_df['best_match_score'] > 0).sum()}")

    export_matrix(data_matrix, feat_meta_df, output_dir=output_dir, prefix="autogcms")

    # ═════════════════════════════════════════════════════════════════════════
    # PASO 3 — Preprocesado estadístico
    # ═════════════════════════════════════════════════════════════════════════
    print("\n─── PASO 3: Preprocesado estadístico ───")
    X_raw = data_matrix.copy()

    ok_min, msg_min = qc.validate_minimum_data(
        X_raw.shape[0], X_raw.shape[1],
        min_samples=getattr(config, 'MIN_SAMPLES_FOR_PCA', 3),
        min_features=getattr(config, 'MIN_FEATURES_FOR_ANALYSIS', 2),
        name="análisis de features",
    )
    if not ok_min:
        print(f"  ⚠ {msg_min}")
        X_proc    = X_raw.copy()
        kept_cols = list(X_raw.columns)
    else:
        X_proc, kept_cols = qc.preprocess_feature_matrix(X_raw, sample_metadata=sample_meta)
        print(f"  Features tras preprocesado: {len(kept_cols)} / {X_raw.shape[1]}")

    # ─── PASO 3a — QC summary ────────────────────────────────────────────────
    print("\n─── PASO 3a: QC summary ───")
    qc_basic = qc.compute_qc_summary(sample_results, gcms_data_dict)
    qc_adv   = qc.compute_advanced_qc_summary(X_raw, X_proc, sample_metadata=sample_meta)
    print(qc_basic.to_string(index=False))

    # ─── PASO 3b — PCA de features ───────────────────────────────────────────
    pca_feat_result = None
    if getattr(config, 'ENABLE_PCA_FEATURES', True) and ok_min:
        print("\n─── PASO 3b: PCA de features ───")
        try:
            ok_pca, msg_pca = qc.validate_minimum_data(
                X_proc.shape[0], X_proc.shape[1],
                min_samples=getattr(config, 'MIN_SAMPLES_FOR_PCA', 3),
                min_features=2,
                name="PCA features",
            )
            if ok_pca:
                pca_feat_result = qc.pca_features(X_proc)
                fig_feat = qc.plot_pca_features(
                    pca_feat_result,
                    sample_metadata=sample_meta,
                    save_path=figures_dir / "pca_features.png",
                )
                if fig_feat:
                    plt.show()
                    plt.close(fig_feat)
                    print(f"  ✅ PCA features mostrado y guardado → {figures_dir / 'pca_features.png'}")
            else:
                print(f"  ⚠ {msg_pca}")
        except Exception:
            print("  ⚠ Error en PCA features:")
            traceback.print_exc()

    # ─── PASO 3c — Tabla de loadings ─────────────────────────────────────────
    loadings_df = pd.DataFrame()
    if pca_feat_result is not None:
        top_n_load  = getattr(config, 'PCA_TOP_LOADINGS_N', 20)
        loadings_df = qc.compute_pca_loadings_table(
            pca_feat_result, feat_meta=feat_meta_df, top_n=top_n_load
        )
        if not loadings_df.empty:
            print(f"\n  Top {top_n_load} loadings (PC1+PC2):")
            print(loadings_df.head(10).to_string(index=False))

    # ─── PASO 3d — Heatmap principal (log-normalizado) ───────────────────────
    if getattr(config, 'ENABLE_HEATMAP_MAIN', True) and ok_min:
        print("\n─── PASO 3d: Heatmap principal ───")
        try:
            ok_hm, msg_hm = qc.validate_minimum_data(
                X_proc.shape[0], X_proc.shape[1],
                min_samples=getattr(config, 'MIN_SAMPLES_FOR_HEATMAP', 2),
                min_features=2,
                name="heatmap principal",
            )
            if ok_hm:
                fig_hm = qc.plot_heatmap_main(
                    X_proc,
                    sample_metadata=sample_meta,
                    top_n=getattr(config, 'HEATMAP_TOP_N', 50),
                    save_path=figures_dir / "heatmap_main.png",
                )
                if fig_hm:
                    plt.show()
                    plt.close(fig_hm)
                    print(f"  ✅ Heatmap principal mostrado y guardado → {figures_dir / 'heatmap_main.png'}")
            else:
                print(f"  ⚠ {msg_hm}")
        except Exception:
            print("  ⚠ Error en heatmap principal:")
            traceback.print_exc()

    # ─── PASO 3e — Heatmap z-score exploratorio ──────────────────────────────
    if getattr(config, 'ENABLE_HEATMAP_ZSCORE', True) and ok_min:
        print("\n─── PASO 3e: Heatmap z-score ───")
        try:
            ok_hz, msg_hz = qc.validate_minimum_data(
                X_proc.shape[0], X_proc.shape[1],
                min_samples=getattr(config, 'MIN_SAMPLES_FOR_HEATMAP', 2),
                min_features=2,
                name="heatmap z-score",
            )
            if ok_hz:
                fig_hz = qc.plot_heatmap_features(
                    X_proc,
                    top_n=getattr(config, 'HEATMAP_TOP_N', 50),
                    sample_metadata=sample_meta,
                    save_path=figures_dir / "heatmap_zscore.png",
                )
                if fig_hz:
                    plt.show()
                    plt.close(fig_hz)
                    print(f"  ✅ Heatmap z-score mostrado y guardado → {figures_dir / 'heatmap_zscore.png'}")
            else:
                print(f"  ⚠ {msg_hz}")
        except Exception:
            print("  ⚠ Error en heatmap z-score:")
            traceback.print_exc()

    # ═════════════════════════════════════════════════════════════════════════
    # PASO 4 — Exportar reporte XLSX consolidado
    # ═════════════════════════════════════════════════════════════════════════
    print("\n─── PASO 4: Exportar reporte XLSX ───")
    try:
        tables = {
            "Data Matrix":      X_raw,
            "Data Matrix Proc": X_proc,
            "Feature Metadata": feat_meta_df,
            "Sample Metadata":  sample_meta,
            "QC Summary Basic": qc_basic,
            "QC Summary Adv":   qc_adv,
        }
        if not loadings_df.empty:
            tables["PCA Loadings"] = loadings_df

        xlsx_path = export_xlsx_report(report_dir, tables, "autogcms_report.xlsx")
        print(f"  Reporte guardado → {xlsx_path}")

        if ok_min and X_proc.shape[1] > 0:
            top_n_list = getattr(config, 'HEATMAP_TOP_N', 50)
            variances  = X_proc.var(axis=0)
            top_feats  = variances.nlargest(min(top_n_list, len(variances))).index.tolist()
            hm_feat_df = feat_meta_df[feat_meta_df["compound_name"].isin(top_feats)].copy()
            hm_feat_path = report_dir / "heatmap_feature_list.xlsx"
            hm_feat_df.to_excel(hm_feat_path, index=False)
            print(f"  Lista de features del heatmap → {hm_feat_path}")

    except Exception:
        print("  ⚠ Error exportando reporte XLSX:")
        traceback.print_exc()

    # ─── Resumen final ────────────────────────────────────────────────────────
    print("\n✔ Pipeline completado.")
    print(f"  Muestras procesadas : {len(sample_results)}")
    print(f"  Features alineadas  : {len(features)}")
    print(f"  Features (proc.)    : {len(kept_cols)}")
    print(f"  Salida              : {output_dir}")


# ── Punto de entrada ──────────────────────────────────────────────────────────
_run_pipeline()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 🔬 VISUALIZACIONES DE CALIDAD: PCAs + Heatmaps
# ─────────────────────────────────────────────────────────────────────────────
# Genera cuatro gráficos sobre los archivos seleccionados en el selector:
#   1) PCA de TIC          — control de calidad técnico del instrumento
#   2) PCA de features     — separación química principal (el PCA del TFG)
#   3) Heatmap log-norm    — tipo paper, top features por varianza
#   4) Heatmap z-score     — exploratorio, patrones relativos
#
# Carga la matriz desde output/autogcms_data_matrix.csv si ya existe
# (generada por la celda 22); si no, la reconstruye desde los CDFs.
# ─────────────────────────────────────────────────────────────────────────────
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import traceback
import IPython
import config
import qc
from pathlib import Path
from preprocessing import preprocess

# ── Parámetros ──────────────────────────────────────────────────────────────
TOP_N_FEATURES  = 40    # Top features para heatmaps (seleccionados por varianza)
MIN_SAMPLES_VIZ = 2     # Mínimo de muestras para PCAs (más laxo que el pipeline)
SAVE_FIGURES    = True  # Guardar PNG en output/figures/

# ── Directorios ─────────────────────────────────────────────────────────────
output_dir  = Path(config.OUTPUT_DIR)
figures_dir = output_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# ── Archivos seleccionados ───────────────────────────────────────────────────
ns = IPython.get_ipython().user_ns
selected_cdf_paths = ns.get('selected_cdf_paths', [])

if not selected_cdf_paths:
    display(HTML(
        '<div style="background:#fff3cd;padding:12px;border-radius:8px;'
        'border:1px solid #ffc107">⚠️ <b>Sin archivos seleccionados.</b> '
        'Usa el selector de la celda 2 primero.</div>'
    ))
    raise SystemExit(0)

sample_names = [Path(p).stem for p in selected_cdf_paths]
sample_meta  = qc.infer_sample_metadata(sample_names)

display(HTML(f'<h3>🔬 Visualizaciones de calidad — {len(sample_names)} muestra(s)</h3>'))
print("Muestras:")
for name in sample_names:
    stype = sample_meta.loc[sample_meta['sample_id'] == name, 'sample_type'].values[0]
    print(f"  • {name}  [{stype}]")

# ══════════════════════════════════════════════════════════════════════════════
# 1) PCA de TIC — Control de calidad del instrumento/inyecciones
# ══════════════════════════════════════════════════════════════════════════════
display(HTML(
    '<hr><h4>📊 1) PCA de TIC — Control de calidad técnico</h4>'
    '<p style="color:#555;font-size:0.9em">'
    'Puntos = muestras/CDFs &nbsp;·&nbsp; Color = tipo (QC / Blank / Sample) &nbsp;·&nbsp; '
    'QC agrupados juntos ✅ = buena reproducibilidad del instrumento</p>'
))

gcms_data_dict = {}
print("Cargando TIC de cada muestra…")
for cdf_path in selected_cdf_paths:
    name = Path(cdf_path).stem
    try:
        gcms_data_dict[name] = preprocess(cdf_path)
        print(f"  ✓ {name}")
    except Exception as exc:
        print(f"  ✗ {name}: {exc}")

if len(gcms_data_dict) >= MIN_SAMPLES_VIZ:
    try:
        tic_matrix, _ = qc.build_tic_matrix(gcms_data_dict)
        pca_tic = qc.pca_tic_qc(tic_matrix)
        if pca_tic is not None:
            evr = pca_tic['explained_variance_ratio']
            pc1 = evr[0] * 100
            pc2 = evr[1] * 100 if len(evr) > 1 else 0.0
            print(f"\n  eje X → PC1 ({pc1:.1f}%)   eje Y → PC2 ({pc2:.1f}%)")
            _save = figures_dir / "pca_tic_qc.png" if SAVE_FIGURES else None
            fig = qc.plot_pca_tic(pca_tic, sample_metadata=sample_meta, save_path=_save)
            if fig:
                plt.show()
                plt.close(fig)
                if SAVE_FIGURES:
                    print(f"  Guardado → {_save}")
        else:
            print("  ⚠ PCA TIC no disponible (necesita >= 2 muestras con datos).")
    except Exception:
        print("  ⚠ Error en PCA TIC:")
        traceback.print_exc()
else:
    print(f"  ⚠ PCA TIC requiere >= {MIN_SAMPLES_VIZ} muestras "
          f"(disponibles: {len(gcms_data_dict)}).")

# ══════════════════════════════════════════════════════════════════════════════
# Cargar / construir la matriz de features alineadas
# ══════════════════════════════════════════════════════════════════════════════
X_raw  = None
X_proc = None

matrix_csv = output_dir / "autogcms_data_matrix.csv"
if matrix_csv.exists():
    try:
        _loaded = pd.read_csv(matrix_csv, sep=';', index_col=0, encoding='utf-8-sig')
        common = [s for s in sample_names if s in _loaded.index]
        if common:
            X_raw = _loaded.loc[common]
            print(f"\nMatriz de features cargada desde CSV: "
                  f"{X_raw.shape[0]} muestras × {X_raw.shape[1]} features")
    except Exception as exc:
        print(f"\n  Error cargando CSV ({exc}), reconstruyendo desde CDFs…")

if X_raw is None:
    print("\nConstruyendo matriz de features desde CDFs…")
    try:
        from peak_detection import find_and_filter_peaks
        from spectra import extract_all_spectra
        from nist_search import match_all_spectra
        from matrix_builder import SampleResult
        from matrix_builder import align_peaks as _ap, build_data_matrix as _bdm

        _lib = ns.get('_nist_library_cache') or ns.get('library') or []
        _results = {}
        for cdf_path in selected_cdf_paths:
            name = Path(cdf_path).stem
            try:
                _data  = gcms_data_dict.get(name) or preprocess(cdf_path)
                _pks   = find_and_filter_peaks(_data)
                _sp    = extract_all_spectra(_data, _pks)
                _mtch  = (match_all_spectra(_sp, _lib, top_n=5, threshold=0.3,
                                            sample_name=name)
                          if _lib else [])
                _results[name] = SampleResult(
                    name=name, peak_table=_pks.table, spectra=_sp, matches=_mtch)
                print(f"  ✓ {name}: {_pks.n_peaks} picos")
            except Exception as exc:
                print(f"  ✗ {name}: {exc}")
        if _results:
            X_raw = _bdm(_ap(_results), list(_results.keys()))
            print(f"  Matriz: {X_raw.shape[0]} × {X_raw.shape[1]}")
    except Exception:
        print("  Error reconstruyendo matriz:")
        traceback.print_exc()

# Preprocesado estadístico
if X_raw is not None and X_raw.shape[0] >= MIN_SAMPLES_VIZ and X_raw.shape[1] >= 2:
    try:
        X_proc, kept_cols = qc.preprocess_feature_matrix(X_raw, sample_metadata=sample_meta)
        print(f"  Features tras preprocesado: {len(kept_cols)} / {X_raw.shape[1]}")
    except Exception as exc:
        print(f"  Error en preprocesado ({exc}), usando matriz cruda.")
        X_proc = X_raw.copy()

# Matriz a usar en heatmaps (preferir preprocesada)
_X_hm = X_proc if X_proc is not None else X_raw

# ══════════════════════════════════════════════════════════════════════════════
# 2) PCA de features/metabolitos — Análisis químico principal
# ══════════════════════════════════════════════════════════════════════════════
display(HTML(
    '<hr><h4>🧪 2) PCA de features/metabolitos — Análisis principal</h4>'
    '<p style="color:#555;font-size:0.9em">'
    'Datos = matriz de features alineadas y preprocesadas &nbsp;·&nbsp; '
    'Puntos = muestras/CDFs &nbsp;·&nbsp; Color = grupo experimental &nbsp;·&nbsp; '
    'Separación en PC1/PC2 → diferencias químicas entre muestras ✅</p>'
))

if X_proc is not None and X_proc.shape[0] >= MIN_SAMPLES_VIZ and X_proc.shape[1] >= 1:
    try:
        pca_feat = qc.pca_features(X_proc)
        if pca_feat is not None:
            evr = pca_feat['explained_variance_ratio']
            pc1 = evr[0] * 100
            pc2 = evr[1] * 100 if len(evr) > 1 else 0.0
            print(f"\n  eje X → PC1 ({pc1:.1f}%)   eje Y → PC2 ({pc2:.1f}%)")
            _save = figures_dir / "pca_features.png" if SAVE_FIGURES else None
            fig = qc.plot_pca_features(
                pca_feat, sample_metadata=sample_meta, save_path=_save)
            if fig:
                plt.show()
                plt.close(fig)
                if SAVE_FIGURES:
                    print(f"  Guardado → {_save}")

            # Tabla de top loadings
            _feat_meta_csv = output_dir / "autogcms_feature_metadata.csv"
            _feat_meta = None
            if _feat_meta_csv.exists():
                try:
                    _feat_meta = pd.read_csv(_feat_meta_csv, sep=';', encoding='utf-8-sig')
                except Exception:
                    pass
            _load_df = qc.compute_pca_loadings_table(
                pca_feat, feat_meta=_feat_meta, top_n=10)
            if not _load_df.empty:
                print("\n  Top 10 loadings PC1 (features más discriminantes):")
                display(_load_df[_load_df['PC'] == 'PC1'].head(10))
    except Exception:
        print("  ⚠ Error en PCA features:")
        traceback.print_exc()
else:
    n = X_proc.shape[0] if X_proc is not None else 0
    print(f"  ⚠ PCA features requiere >= {MIN_SAMPLES_VIZ} muestras (disponibles: {n}).")

# ══════════════════════════════════════════════════════════════════════════════
# 3) Heatmap principal — tipo paper (abundancia log-normalizada)
# ══════════════════════════════════════════════════════════════════════════════
display(HTML(
    '<hr><h4>🗺️ 3) Heatmap principal — tipo paper</h4>'
    '<p style="color:#555;font-size:0.9em">'
    f'Filas = top {TOP_N_FEATURES} features por varianza &nbsp;·&nbsp; '
    'Columnas = muestras ordenadas por tipo &nbsp;·&nbsp; '
    'Color = abundancia log-normalizada (YlOrRd) &nbsp;·&nbsp; '
    'Orden columnas: QC → Blank → Control → Sample</p>'
))

if _X_hm is not None and _X_hm.shape[0] >= 2 and _X_hm.shape[1] >= 2:
    try:
        _save = figures_dir / "heatmap_main.png" if SAVE_FIGURES else None
        fig = qc.plot_heatmap_main(
            _X_hm, sample_metadata=sample_meta,
            top_n=TOP_N_FEATURES, save_path=_save,
        )
        if fig:
            plt.show()
            plt.close(fig)
            if SAVE_FIGURES:
                print(f"  Guardado → {_save}")
    except Exception:
        print("  ⚠ Error en heatmap principal:")
        traceback.print_exc()
else:
    print("  ⚠ Heatmap requiere >= 2 muestras y >= 2 features.")

# ══════════════════════════════════════════════════════════════════════════════
# 4) Heatmap z-score — exploratorio
# ══════════════════════════════════════════════════════════════════════════════
display(HTML(
    '<hr><h4>📉 4) Heatmap z-score — exploratorio</h4>'
    '<p style="color:#555;font-size:0.9em">'
    'Igual que el heatmap principal pero con z-score por feature. '
    'Resalta patrones relativos entre muestras. '
    'Paleta divergente: azul = bajo, rojo = alto respecto a la media.</p>'
))

if _X_hm is not None and _X_hm.shape[0] >= 2 and _X_hm.shape[1] >= 2:
    try:
        _save = figures_dir / "heatmap_zscore.png" if SAVE_FIGURES else None
        fig = qc.plot_heatmap_features(
            _X_hm, top_n=TOP_N_FEATURES,
            sample_metadata=sample_meta, save_path=_save,
        )
        if fig:
            plt.show()
            plt.close(fig)
            if SAVE_FIGURES:
                print(f"  Guardado → {_save}")
    except Exception:
        print("  ⚠ Error en heatmap z-score:")
        traceback.print_exc()
else:
    print("  ⚠ Heatmap z-score requiere >= 2 muestras y >= 2 features.")

# ── Resumen ──────────────────────────────────────────────────────────────────
display(HTML('<hr><p style="color:green;font-weight:bold">✅ Visualizaciones completadas.</p>'))
if SAVE_FIGURES:
    print(f"Figuras guardadas en: {figures_dir}")


---
## Ajuste de parametros

Para procesar otra muestra individual, vuelve a la seccion 1 y cambia `FILE_IDX`.
La seccion 6 siempre procesa **todas** las muestras y genera el reporte completo.

Si necesitas ajustar la sensibilidad, edita `config.py`:

| Parametro | Efecto al aumentar |
|---|---|
| `PEAK_MIN_PROMINENCE` | Menos picos (solo los mas intensos) |
| `PEAK_MIN_DISTANCE` | Menos picos cercanos entre si |
| `PEAK_SNR_THRESHOLD` | Elimina picos con mas ruido |
| `MAX_PEAKS` | Limita el numero total de picos (None = sin limite) |
| `SAVGOL_WINDOW` | Mas suavizado (puede perder picos estrechos) |
| `BASELINE_LAMBDA` | Baseline mas suave / rigida |
| `APEX_SCANS_AVERAGE` | Mas scans promediados (mas robusto, menos resolucion) |
| `COELUTION_BIC_DELTA` | Menos co-eluciones detectadas (criterio mas estricto) |
| `SPECTRA_AVG_SCANS` | Scans promediados en subpicos co-eluidos |
| `TIC_BIN_SIZE_MIN` | Resolucion del binning para PCA QC |
| `COSINE_THRESHOLD` | Menos identificaciones pero mas fiables |

In [ ]:
# DEBUG: listar archivos de salida
from pathlib import Path
import config

report_dir = Path(config.OUTPUT_DIR) / "report"
figures_dir = Path(config.OUTPUT_DIR) / "figures"

print(f"Directorio de reportes: {report_dir}")
if report_dir.exists():
    files = sorted(report_dir.glob('*'))
    if files:
        print(f"Archivos en {report_dir.name}:")
        for f in files:
            size = f"({f.stat().st_size / 1024:.1f} KB)" if f.is_file() else "(directorio)"
            print(f"  • {f.name} {size}")
    else:
        print("  (vacío)")
else:
    print("  [No existe aún]")

print(f"\nDirectorio de figuras: {figures_dir}")
if figures_dir.exists():
    figures = sorted(figures_dir.glob('*.png'))
    if figures:
        print(f"Figuras en {figures_dir.name}:")
        for f in figures:
            print(f"  • {f.name}")
    else:
        print("  (vacío)")
else:
    print("  [No existe aún]")

In [ ]:
# Mostrar logs si existen
from pathlib import Path
import config

report_dir = Path(config.OUTPUT_DIR) / "report"

log_files = list(report_dir.glob('*.log')) if report_dir.exists() else []

if log_files:
    print(f"Archivos de log en {report_dir}:")
    for log_file in log_files:
        print(f"\n📄 {log_file.name}:")
        print("-" * 80)
        try:
            with open(log_file, 'r', encoding='utf-8') as f:
                content = f.read()
                print(content[:2000])  # Primeros 2000 caracteres
                if len(content) > 2000:
                    print(f"\n... ({len(content) - 2000} caracteres más) ...")
        except Exception as e:
            print(f"Error leyendo: {e}")
else:
    print(f"No hay archivos .log en {report_dir}")

In [ ]:
# Verificar cada cdf en cdf_files y si existe en disco
from pathlib import Path
try:
    print('cdf_files:')
    for p in cdf_files:
        pp = Path(p)
        print(' -', pp, 'exists=', pp.exists())
except NameError:
    print('cdf_files no definido en este scope')


In [ ]:
# Comprobar dependencias Python requeridas y sugerir `pip install` si faltan
import importlib, sys
modules = ['numpy','pandas','netCDF4','seaborn','scipy','sklearn','matplotlib','openpyxl','ipywidgets']
missing = [m for m in modules if importlib.util.find_spec(m) is None]
print('Missing modules:', missing)
if missing:
    print('\nInstalación sugerida (ejecuta en PowerShell):')
    for pkg in missing:
        print(f"{sys.executable} -m pip install {pkg}")

# También comprobar si el módulo local `pipeline` es importable
try:
    import pipeline
    print('\nlocal module pipeline: OK')
except Exception as e:
    print('\nlocal module pipeline: ERROR ->', type(e).__name__, e)


In [ ]:
# Intentar añadir el root del proyecto a sys.path y reimportar `pipeline`
import sys, os
cwd = os.getcwd()
candidates = [os.path.abspath(os.path.join(cwd, '..')), os.path.abspath(os.path.join(cwd, '..', '..'))]
added = []
for p in candidates:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
        added.append(p)
print('Paths añadidos a sys.path:', added)
try:
    import pipeline
    print('Import pipeline: OK ->', getattr(pipeline, '__file__', '?'))
except Exception as e:
    print('Import pipeline: ERROR ->', type(e).__name__, e)
